# 09 · DeBERTa-v3 con ajuste fino

> **Viene de:** `08_features_embeddings` · **Lee:** `summaries_train_clean.csv`, `prompts_train_clean.csv` y, si está disponible, `08_oof_predictions.csv` · **Produce:** `09_deberta_oof_predictions.csv`, `09_deberta_results.csv`, `09_deberta_by_prompt.csv`, `09_deberta_fold_log.csv`, `09_deberta_pred_vs_real.png` · **Sigue con:** 10, comparación final (pendiente)

**Objetivo:** probar el último nivel de la escalera de modelos: un transformer **ajustado a la tarea** (fine-tuning).
En el notebook 08, los embeddings genéricos (spaCy y Sentence-BERT) no superaron a la media por sí solos, porque
miden **de qué trata** el texto y no **qué tan bueno es**. Aquí el modelo aprende directamente a predecir
`content` y `wording`, así que sus representaciones se ajustan a lo que distingue un buen resumen.

| Elemento | Decisión |
|---|---|
| Modelo | `microsoft/deberta-v3-base` (~184 M de parámetros) |
| Entrada | `prompt_question [SEP] resumen`, hasta 512 tokens |
| Salida | Mean pooling + capa lineal con 2 salidas (`content`, `wording`), pérdida MSE |
| Validación | `LeaveOneGroupOut` por `prompt_id`, los mismos 4 folds que 07 y 08 |
| Hiperparámetros | Fijados **antes** de ver resultados; sin early stopping |

**Por qué sin early stopping:** elegir la época mirando el error del prompt de validación sería usar ese prompt
para tomar decisiones del modelo (fuga de información). Se entrena un número fijo de épocas, igual que en 08 se
fijaron los parámetros de LightGBM de antemano.

**Dónde se ejecuta:** en Google Colab con GPU. El notebook detecta si está en Colab (usa una carpeta de Google
Drive) o en el repositorio local (usa `data/` y `outputs/`). Si la sesión se corta, al volver a ejecutarlo se
saltan los folds que ya terminaron.

## 1. Entorno, rutas y configuración

In [ ]:
import gc
import json
import os
import random
import sys
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # El tokenizador de DeBERTa-v3 usa SentencePiece.
    get_ipython().run_line_magic("pip", "install -q sentencepiece protobuf")
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto2_DS_deberta")
    INPUT_DIR = BASE_DIR / "data"
    OUTPUT_DIR = BASE_DIR / "outputs"
    PROCESSED_DIR = TABLES_DIR = FIGURES_DIR = OUTPUT_DIR
    OOF_08_PATH = INPUT_DIR / "08_oof_predictions.csv"
else:
    PROJECT_ROOT = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / "src").is_dir() and (path / "notebooks").is_dir()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("No se encontró la raíz del proyecto.")
    INPUT_DIR = PROJECT_ROOT / "data" / "interim"
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
    TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
    OOF_08_PATH = PROCESSED_DIR / "08_oof_predictions.csv"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import LeaveOneGroupOut
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

print("Colab:", IN_COLAB, "| entrada:", INPUT_DIR)

**Sintaxis:**

* `try: import google.colab ... except ImportError:` es la forma estándar de saber si el código corre en Colab:
  ese módulo solo existe ahí.
* `get_ipython().run_line_magic("pip", "...")` es lo mismo que escribir `%pip install ...` en una celda, pero
  dentro de un `if`, para que en local no intente instalar nada.
* `drive.mount(...)` conecta Google Drive como si fuera una carpeta más. Así los resultados sobreviven aunque
  Colab cierre la sesión (lo que se guarda en `/content` se borra).

In [ ]:
# ---- Prueba corta: True para comprobar que todo funciona antes de la corrida completa ----
SMOKE_TEST: bool = False

CONFIG: dict = dict(
    model_name="microsoft/deberta-v3-base",
    max_len=512,
    epochs=3,
    batch_size=8,
    lr_encoder=2e-5,      # tasa de aprendizaje de DeBERTa (pesos ya preentrenados: pasos pequeños)
    lr_head=1e-3,         # tasa de la capa nueva de regresión (empieza desde cero: pasos más grandes)
    weight_decay=0.01,
    warmup_ratio=0.10,
    max_grad_norm=1.0,
    seed=42,
)
if SMOKE_TEST:
    CONFIG.update(epochs=1, smoke_rows=400, smoke_folds=1)

TARGETS: list[str] = ["content", "wording"]
MODEL_LABEL: str = "DeBERTa-v3-base (fine-tuning)"

# En la prueba corta las salidas van a una subcarpeta para no mezclarse con la corrida real.
RUN_DIR: Path = (PROCESSED_DIR / "09_smoke_test") if SMOKE_TEST else PROCESSED_DIR
FOLDS_DIR: Path = RUN_DIR / "09_deberta_folds"
OUT_TABLES: Path = (RUN_DIR if SMOKE_TEST else TABLES_DIR)
OUT_FIGURES: Path = (RUN_DIR if SMOKE_TEST else FIGURES_DIR)
for directory in [FOLDS_DIR, OUT_TABLES, OUT_FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP: bool = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Sin GPU: el entrenamiento completo sería demasiado lento. En Colab: Entorno de ejecución → "
          "Cambiar tipo de entorno de ejecución → GPU T4.")
print(json.dumps(CONFIG, indent=1))


def set_seed(seed: int) -> None:
    # Fija todas las fuentes de azar para que dos ejecuciones den resultados parecidos.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

* `CONFIG.update(...)` cambia solo algunas llaves del diccionario y conserva las demás.
* **Dos tasas de aprendizaje:** DeBERTa ya sabe mucho de inglés, así que sus pesos se mueven poco (`2e-5`); la capa
  de regresión es nueva y aleatoria, así que se le permite aprender más rápido (`1e-3`).
* `USE_AMP` activa la **precisión mixta** (fp16) solo con GPU: los cálculos se hacen con números de 16 bits donde es
  seguro, lo que ahorra memoria y casi duplica la velocidad en una T4.

## 2. Datos

Se usan las mismas copias limpias que en todos los notebooks anteriores. A cada resumen se le agrega la pregunta
de su prompt (`prompt_question`), que le dice al modelo qué se pedía resumir.

In [ ]:
summaries: pd.DataFrame = pd.read_csv(INPUT_DIR / "summaries_train_clean.csv")
prompts: pd.DataFrame = pd.read_csv(INPUT_DIR / "prompts_train_clean.csv")
data: pd.DataFrame = summaries.merge(prompts[["prompt_id", "prompt_question", "prompt_title"]],
                                     on="prompt_id", how="left", validate="many_to_one", sort=False)

assert len(data) == 7165 and data["student_id"].is_unique
assert data["prompt_id"].nunique() == 4
assert data[["text", "prompt_question", *TARGETS]].notna().all().all()
assert data["student_id"].tolist() == summaries["student_id"].tolist()   # mismo orden que en 08

if SMOKE_TEST:
    data = (data.groupby("prompt_id").sample(n=CONFIG["smoke_rows"] // 4, random_state=CONFIG["seed"])
                .reset_index(drop=True))

y: np.ndarray = data[TARGETS].to_numpy(dtype=np.float32)
groups: np.ndarray = data["prompt_id"].to_numpy()
print("Filas:", len(data), "| prompts:", sorted(data["prompt_id"].unique()))
display(data[["student_id", "prompt_title", "prompt_question", "text", *TARGETS]].head(3))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

# Largo de cada entrada en tokens de DeBERTa, sin recortar, para saber cuántas exceden max_len
lengths: np.ndarray = np.array([
    len(ids) for ids in tokenizer(data["prompt_question"].tolist(), data["text"].tolist())["input_ids"]
])
print(f"Tokens por entrada: mediana={np.median(lengths):.0f}, p95={np.percentile(lengths, 95):.0f}, "
      f"máximo={lengths.max()}")
print(f"Entradas que exceden {CONFIG['max_len']} tokens: {(lengths > CONFIG['max_len']).mean():.2%}")

* El tokenizador de DeBERTa parte las palabras en **subpalabras** (por ejemplo, `summarizing` → `▁summar`, `izing`),
  así que el número de tokens es mayor que el número de palabras de `tokenize_text`.
* Si casi ninguna entrada pasa de 512 tokens, recortar en 512 no pierde información relevante.

## 3. Dataset y modelo

```
prompt_question [SEP] resumen  →  DeBERTa-v3  →  un vector por token  →  promedio (mean pooling)  →  Linear  →  [content, wording]
```

* **Mean pooling:** DeBERTa devuelve un vector por token; se promedian (ignorando el relleno) para tener un solo
  vector por resumen.
* La capa lineal final convierte ese vector en las dos puntuaciones.

In [ ]:
class SummaryDataset(Dataset):
    # Entrega cada resumen ya tokenizado junto con sus dos puntuaciones.

    def __init__(self, frame: pd.DataFrame, targets: np.ndarray | None = None):
        self.encodings = tokenizer(frame["prompt_question"].tolist(), frame["text"].tolist(),
                                   truncation="only_second", max_length=CONFIG["max_len"])
        self.targets = targets

    def __len__(self) -> int:
        return len(self.encodings["input_ids"])

    def __getitem__(self, index: int) -> dict:
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.targets is not None:
            item["labels"] = self.targets[index].tolist()
        return item


class MeanPoolingRegressor(nn.Module):
    # DeBERTa + promedio de los vectores de los tokens + capa lineal con 2 salidas.

    def __init__(self, model_name: str, n_outputs: int = 2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.encoder.config.hidden_size, n_outputs)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                              token_type_ids=token_type_ids).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return self.head(pooled)


collator = DataCollatorWithPadding(tokenizer)

**Sintaxis:**

* `class SummaryDataset(Dataset)`: PyTorch pide dos métodos, `__len__` (cuántos ejemplos hay) y `__getitem__`
  (devolver el ejemplo número `index`). El `DataLoader` los usa para armar lotes.
* `truncation="only_second"` recorta solo el **resumen** (el segundo texto) si la entrada es muy larga; la pregunta
  del prompt siempre queda completa.
* `DataCollatorWithPadding` rellena con ceros los resúmenes de cada lote hasta el más largo **de ese lote**, no hasta
  512. Eso ahorra mucho tiempo porque la mayoría de los resúmenes son cortos.
* `attention_mask.unsqueeze(-1)` agrega una dimensión al final para poder multiplicar la máscara `(lote, tokens)`
  por los vectores `(lote, tokens, 768)`. Los tokens de relleno tienen máscara 0 y no cuentan en el promedio.
* `.clamp(min=1e-9)` evita dividir entre cero.

## 4. Entrenamiento con Leave-One-Prompt-Out

Para cada fold se entrena un modelo **nuevo** con 3 prompts y se predice el cuarto. Al terminar cada fold, sus
predicciones se guardan en un archivo propio; si la sesión se corta, al volver a ejecutar esta celda los folds
terminados se cargan del disco en vez de entrenarse otra vez.

In [ ]:
def make_loader(frame: pd.DataFrame, targets: np.ndarray | None, shuffle: bool) -> DataLoader:
    return DataLoader(SummaryDataset(frame, targets), batch_size=CONFIG["batch_size"],
                      shuffle=shuffle, collate_fn=collator)


def to_device(batch: dict) -> dict:
    return {key: value.to(DEVICE) for key, value in batch.items()}


def train_one_fold(train_frame: pd.DataFrame, train_targets: np.ndarray) -> tuple[nn.Module, list[float]]:
    set_seed(CONFIG["seed"])
    model = MeanPoolingRegressor(CONFIG["model_name"], n_outputs=len(TARGETS)).to(DEVICE)
    loader = make_loader(train_frame, train_targets, shuffle=True)

    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": CONFIG["lr_encoder"]},
        {"params": model.head.parameters(), "lr": CONFIG["lr_head"]},
    ], weight_decay=CONFIG["weight_decay"])
    total_steps = len(loader) * CONFIG["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(CONFIG["warmup_ratio"] * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loss_fn = nn.MSELoss()

    epoch_losses: list[float] = []
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        running, start = 0.0, time.perf_counter()
        for step, batch in enumerate(loader, start=1):
            batch = to_device(batch)
            labels = batch.pop("labels")
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                predictions = model(**batch)
            loss = loss_fn(predictions.float(), labels.float())

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running += loss.item()
            if step % 100 == 0 or step == len(loader):
                print(f"  época {epoch} · paso {step}/{len(loader)} · MSE promedio {running / step:.4f}", flush=True)
        epoch_losses.append(running / len(loader))
        print(f"  época {epoch} terminada en {time.perf_counter() - start:.0f} s", flush=True)
    return model, epoch_losses


@torch.no_grad()
def predict(model: nn.Module, frame: pd.DataFrame) -> np.ndarray:
    model.eval()
    outputs = []
    for batch in make_loader(frame, None, shuffle=False):
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs.append(model(**to_device(batch)).float().cpu().numpy())
    return np.vstack(outputs)

**Sintaxis:**

* `AdamW([{...}, {...}])`: cada diccionario es un **grupo de parámetros** con su propia tasa de aprendizaje.
* `get_linear_schedule_with_warmup` sube la tasa de aprendizaje desde 0 durante el primer 10% de pasos (warmup) y
  luego la baja linealmente hasta 0. Evita pasos bruscos al inicio, cuando la capa nueva todavía es aleatoria.
* `torch.amp.autocast(...)` + `GradScaler`: precisión mixta. El `scaler` multiplica la pérdida antes de
  `backward()` para que los gradientes pequeños no se vuelvan cero en 16 bits, y la divide de nuevo antes de
  actualizar los pesos (`unscale_`).
* `clip_grad_norm_` limita el tamaño total del gradiente a 1.0 para que un lote raro no desestabilice el modelo.
* `@torch.no_grad()` desactiva el cálculo de gradientes al predecir: más rápido y con menos memoria.
* `model.train()` / `model.eval()` activan o desactivan el *dropout*, que solo debe usarse al entrenar.

In [ ]:
logo = LeaveOneGroupOut()
folds = list(logo.split(data, y, groups))
if SMOKE_TEST:
    folds = folds[:CONFIG["smoke_folds"]]

fold_log_path: Path = OUT_TABLES / "09_deberta_fold_log.csv"
fold_log: list[dict] = pd.read_csv(fold_log_path).to_dict("records") if fold_log_path.exists() else []

for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
    prompt_id = groups[valid_idx][0]
    fold_path = FOLDS_DIR / f"09_deberta_fold_{prompt_id}.csv"
    if fold_path.exists():
        print(f"Fold {fold} ({prompt_id}): ya existe, se carga de {fold_path.name}")
        continue

    print(f"Fold {fold} ({prompt_id}, {data.loc[valid_idx[0], 'prompt_title']}): "
          f"entrenamiento={len(train_idx)}, validación={len(valid_idx)}", flush=True)
    start = time.perf_counter()
    model, epoch_losses = train_one_fold(data.iloc[train_idx], y[train_idx])
    prediction = predict(model, data.iloc[valid_idx])
    seconds = time.perf_counter() - start

    fold_frame = data.iloc[valid_idx][["student_id", "prompt_id"]].copy()
    fold_frame["pred_content"], fold_frame["pred_wording"] = prediction[:, 0], prediction[:, 1]
    fold_frame.to_csv(fold_path, index=False)

    fold_rmse = [root_mean_squared_error(y[valid_idx, j], prediction[:, j]) for j in range(len(TARGETS))]
    fold_log.append({"fold": fold, "prompt_id": prompt_id, "segundos": seconds,
                     "mse_entrenamiento_ultima_epoca": epoch_losses[-1],
                     "rmse_content": fold_rmse[0], "rmse_wording": fold_rmse[1],
                     "mcrmse": float(np.mean(fold_rmse))})
    pd.DataFrame(fold_log).to_csv(fold_log_path, index=False)
    print(f"Fold {fold} listo en {seconds / 60:.1f} min · MCRMSE {np.mean(fold_rmse):.4f}", flush=True)

    del model
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame(fold_log).round(4))

* Cada fold guarda `09_deberta_fold_<prompt_id>.csv` apenas termina. Si Colab se desconecta, basta con volver a
  ejecutar todas las celdas: `if fold_path.exists(): continue` salta lo ya hecho.
* `del model`, `gc.collect()` y `torch.cuda.empty_cache()` liberan la memoria de la GPU antes del siguiente fold.
* El MSE de entrenamiento se registra solo para revisar que el modelo aprende; **no** se usa para elegir nada.

## 5. Resultados out-of-fold

Se juntan las predicciones de los folds. Cada resumen fue predicho por un modelo que nunca vio su prompt, igual que
en 07 y 08, así que las métricas son directamente comparables.

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    # Misma fórmula que src/evaluation.regression_metrics (aquí se copia para no depender del repositorio en Colab).
    metrics: dict[str, float] = {}
    rmse_values = []
    for j, target in enumerate(TARGETS):
        rmse = root_mean_squared_error(y_true[:, j], y_pred[:, j])
        rmse_values.append(rmse)
        metrics[f"rmse_{target}"] = rmse
        metrics[f"mae_{target}"] = mean_absolute_error(y_true[:, j], y_pred[:, j])
        metrics[f"r2_{target}"] = r2_score(y_true[:, j], y_pred[:, j])
    metrics["mcrmse"] = float(np.mean(rmse_values))
    return metrics


fold_files = sorted(FOLDS_DIR.glob("09_deberta_fold_*.csv"))
predicted = pd.concat([pd.read_csv(path) for path in fold_files], ignore_index=True)
assert predicted["student_id"].is_unique
if not SMOKE_TEST:
    assert len(fold_files) == 4 and len(predicted) == len(data), "Faltan folds: vuelva a ejecutar la sección 4."

oof = (data[["student_id", "prompt_id", "prompt_title", *TARGETS]]
       .merge(predicted.drop(columns="prompt_id"), on="student_id", how="inner", validate="one_to_one"))
oof["modelo"] = MODEL_LABEL
for target in TARGETS:
    oof[f"error_abs_{target}"] = (oof[target] - oof[f"pred_{target}"]).abs()

y_oof = oof[TARGETS].to_numpy()
p_oof = oof[["pred_content", "pred_wording"]].to_numpy()
results = pd.DataFrame([{"modelo": MODEL_LABEL, "filas": len(oof), **regression_metrics(y_oof, p_oof)}])
by_prompt = pd.DataFrame([
    {"prompt_id": prompt_id, "prompt_title": g["prompt_title"].iloc[0], "filas": len(g),
     **regression_metrics(g[TARGETS].to_numpy(), g[["pred_content", "pred_wording"]].to_numpy())}
    for prompt_id, g in oof.groupby("prompt_id")
])
display(results.round(4))
display(by_prompt.round(4))

columns_08 = ["student_id", "prompt_id", *TARGETS, "modelo", "pred_content", "error_abs_content",
              "pred_wording", "error_abs_wording"]
oof[columns_08].to_csv(RUN_DIR / "09_deberta_oof_predictions.csv", index=False)
results.to_csv(OUT_TABLES / "09_deberta_results.csv", index=False)
by_prompt.to_csv(OUT_TABLES / "09_deberta_by_prompt.csv", index=False)

* `validate="one_to_one"` hace que `merge` falle si algún `student_id` aparece repetido: es una verificación gratis.
* `09_deberta_oof_predictions.csv` tiene **las mismas columnas** que `08_oof_predictions.csv` y
  `07_tfidf_oof_predictions.csv`, para que la comparación final (notebook 10) pueda juntarlos directamente.

## 6. Comparación con el notebook 08 y promedio fijo 50/50

Si `08_oof_predictions.csv` está disponible (en Colab, subirlo a la carpeta `data/` de Drive), se compara DeBERTa
con los modelos de 08 **en los mismos resúmenes**. También se prueba el promedio simple de DeBERTa y la fusión
mínima de 08. El peso 50/50 se fijó **antes** de ver los resultados: elegirlo mirando el error OOF sería ajustar
un parámetro con los datos de evaluación.

In [ ]:
REFERENCE_MODELS: list[str] = ["Solo largo + Ridge", "A. Features + LightGBM",
                               "Opción 1: Fusión mínima (A + cos_sim) + LightGBM"]
BLEND_BASE: str = "Opción 1: Fusión mínima (A + cos_sim) + LightGBM"

if OOF_08_PATH.exists():
    oof_08 = pd.read_csv(OOF_08_PATH)
    oof_08 = oof_08[oof_08["student_id"].isin(oof["student_id"])]
    rows = []
    for name in REFERENCE_MODELS:
        ref = oof[["student_id"]].merge(oof_08[oof_08["modelo"] == name], on="student_id", validate="one_to_one")
        rows.append({"modelo": name, **regression_metrics(ref[TARGETS].to_numpy(),
                                                          ref[["pred_content", "pred_wording"]].to_numpy())})
    rows.append({"modelo": MODEL_LABEL, **regression_metrics(y_oof, p_oof)})

    base = oof[["student_id"]].merge(oof_08[oof_08["modelo"] == BLEND_BASE], on="student_id", validate="one_to_one")
    blend = 0.5 * p_oof + 0.5 * base[["pred_content", "pred_wording"]].to_numpy()
    rows.append({"modelo": "Promedio 50/50: DeBERTa + fusión mínima", **regression_metrics(y_oof, blend)})

    comparison = pd.DataFrame(rows).sort_values("mcrmse").reset_index(drop=True)
    display(comparison.round(4))
    comparison.to_csv(OUT_TABLES / "09_deberta_comparison.csv", index=False)
else:
    print(f"No se encontró {OOF_08_PATH}; se omite la comparación con 08.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=[11, 4.6])
for j, (ax, target) in enumerate(zip(axes, TARGETS)):
    ax.scatter(y_oof[:, j], p_oof[:, j], s=7, alpha=0.2, color="#3978a8")
    limits = [min(y_oof[:, j].min(), p_oof[:, j].min()), max(y_oof[:, j].max(), p_oof[:, j].max())]
    ax.plot(limits, limits, color="#777777", linestyle="--", linewidth=1)
    ax.set(title=f"{target} · RMSE {root_mean_squared_error(y_oof[:, j], p_oof[:, j]):.3f}",
           xlabel="Puntuación real", ylabel="Predicción out-of-fold")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(MODEL_LABEL)
fig.tight_layout()
fig.savefig(OUT_FIGURES / "09_deberta_pred_vs_real.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Conclusiones

*Completar después de la corrida completa, con los números de las tablas de arriba.* Preguntas a responder:

1. ¿DeBERTa supera a las features manuales (MCRMSE 0.581) y a la fusión mínima (0.576) con prompts no vistos?
2. ¿La mejora está en `content`, en `wording` o en los dos? En 08, `wording` era el objetivo difícil (RMSE 0.67).
3. ¿El promedio 50/50 mejora a los dos modelos por separado? Si es así, las features y el transformer capturan
   información distinta.
4. ¿La mejora se repite en los 4 prompts o depende de uno solo?
5. ¿Cuánto tarda frente a LightGBM (segundos contra decenas de minutos)?

## Puente al siguiente notebook

Las predicciones out-of-fold de DeBERTa quedan en `09_deberta_oof_predictions.csv`, con el mismo formato que las de
07 y 08. El notebook 10 (pendiente) juntará los tres archivos en una sola tabla, elegirá los finalistas y ajustará
sus hiperparámetros con validación agrupada por prompt.